# Experimento 3

In [26]:

from google.colab import drive
import os

drive.mount('/content/drive')

DATASET_DIR = "/content/drive/MyDrive/data_seed_ajustado"

if os.path.exists(DATASET_DIR):
    print("Dataset encontrado correctamente.")
    print("Ruta:", DATASET_DIR)
    print("Carpetas:", os.listdir(DATASET_DIR)[:10])
else:
    print("No se encontró el dataset en la ruta indicada.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset encontrado correctamente.
Ruta: /content/drive/MyDrive/data_seed_ajustado
Carpetas: ['Achnatherum inebrians', 'Agriophyllum squarrosum', 'Agropyron mongolicum', 'Achnatherum splendens', 'Agropyron cristatum', 'Anemone rivularis', 'Agropyron elongatum', 'Amorpha fruticosa', 'Amygdalus mongolica', 'Apocynum venetum']


In [27]:

from collections import Counter

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

class_counts = {}

for class_name in sorted(os.listdir(DATASET_DIR)):
    class_path = os.path.join(DATASET_DIR, class_name)

    if os.path.isdir(class_path):
        images = [
            file for file in os.listdir(class_path)
            if file.lower().endswith(image_extensions)
        ]
        class_counts[class_name] = len(images)

total_classes = len(class_counts)
total_images = sum(class_counts.values())

print(f"Número de especies: {total_classes}")
print(f"Número total de imágenes: {total_images}")
print(f"Carpetas sin imágenes: {sum(1 for count in class_counts.values() if count == 0)}")

print("\nPrimeras 15 especies:")
for species, count in list(class_counts.items())[:15]:
    print(f"{species}: {count} imágenes")

Número de especies: 88
Número total de imágenes: 4496
Carpetas sin imágenes: 0

Primeras 15 especies:
Achnatherum inebrians: 42 imágenes
Achnatherum splendens: 43 imágenes
Agriophyllum squarrosum: 89 imágenes
Agropyron cristatum: 56 imágenes
Agropyron elongatum: 37 imágenes
Agropyron mongolicum: 47 imágenes
Amorpha fruticosa: 40 imágenes
Amygdalus mongolica: 40 imágenes
Anemone rivularis: 49 imágenes
Apocynum pictum: 45 imágenes
Apocynum venetum: 49 imágenes
Artemisia desertorum: 45 imágenes
Artemisia ordosica: 49 imágenes
Artemisia sphaerocephala: 76 imágenes
Astragalus laxmannii: 37 imágenes


In [28]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria GPU:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

Dispositivo: cuda
GPU: Tesla T4
Memoria GPU: 14.56 GB


In [29]:
!pip install -q open_clip_torch

In [30]:
import open_clip

model, _, preprocess = open_clip.create_model_and_transforms(
    'hf-hub:imageomics/bioclip'
)

model = model.to(device)

print("BioCLIP cargado correctamente.")

BioCLIP cargado correctamente.


In [31]:
classes = sorted(class_counts.keys())

class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(classes)
}

idx_to_class = {
    idx: class_name
    for class_name, idx in class_to_idx.items()
}

print("Número de clases:", len(classes))
print("Ejemplos:")

for i in range(5):
    print(f"{i}: {classes[i]}")

Número de clases: 88
Ejemplos:
0: Achnatherum inebrians
1: Achnatherum splendens
2: Agriophyllum squarrosum
3: Agropyron cristatum
4: Agropyron elongatum


In [32]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

dataset = ImageFolder(
    root=DATASET_DIR,
    transform=preprocess
)

print("Imágenes:", len(dataset))
print("Clases:", len(dataset.classes))
print("Primeras clases:", dataset.classes[:5])

Imágenes: 4496
Clases: 88
Primeras clases: ['Achnatherum inebrians', 'Achnatherum splendens', 'Agriophyllum squarrosum', 'Agropyron cristatum', 'Agropyron elongatum']


In [33]:
from sklearn.model_selection import train_test_split
import numpy as np

indices = np.arange(len(dataset))
labels = np.array(dataset.targets)

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=labels[temp_idx],
    random_state=42
)

print("Entrenamiento:", len(train_idx))
print("Validación:", len(val_idx))
print("Prueba:", len(test_idx))

Entrenamiento: 3147
Validación: 674
Prueba: 675


In [43]:
from torch.utils.data import Subset

# Definimos los subsets utilizando los índices estratificados generados previamente
train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("DataLoaders creados y actualizados correctamente.")

DataLoaders creados y actualizados correctamente.


In [35]:
print(model.visual)

VisionTransformer(
  (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
  (patch_dropout): Identity()
  (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  (transformer): Transformer(
    (resblocks): ModuleList(
      (0-11): 12 x ResidualAttentionBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (ls_1): Identity()
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): Sequential(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
        (ls_2): Identity()
      )
    )
  )
  (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
)


# Configurar el fine-tuning

In [36]:
import torch.nn as nn

# Congelar todo BioCLIP
for param in model.parameters():
    param.requires_grad = False

# Descongelar los últimos 2 bloques visuales
for block in model.visual.transformer.resblocks[-2:]:
    for param in block.parameters():
        param.requires_grad = True

# Descongelar la normalización final
for param in model.visual.ln_post.parameters():
    param.requires_grad = True

# Clasificador para las 88 especies
model.classifier = nn.Linear(768, len(classes)).to(device)

# El clasificador siempre será entrenable
for param in model.classifier.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

total_params = sum(p.numel() for p in model.parameters())

print(f"Parámetros totales: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")

Parámetros totales: 149,688,409
Parámetros entrenables: 14,244,952


# Configurar entrenamiento

In [37]:
optimizer = torch.optim.AdamW(
    [
        {
            "params": model.visual.transformer.resblocks[-2:].parameters(),
            "lr": 1e-5
        },
        {
            "params": model.visual.ln_post.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.classifier.parameters(),
            "lr": 1e-3
        }
    ],
    weight_decay=1e-4
)

print("Optimizador actualizado correctamente.")

Optimizador actualizado correctamente.


In [38]:
import torch.nn as nn

# Definimos la función de pérdida para clasificación multiclase
criterion = nn.CrossEntropyLoss()
print("Función de pérdida (criterion) definida correctamente.")

Función de pérdida (criterion) definida correctamente.


# Función de entrenamiento y validación

In [39]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

In [44]:
model.classifier = nn.Linear(768, len(classes)).to(device)

for param in model.classifier.parameters():
    param.requires_grad = True

print("Clasificador corregido (dimensión 768):", model.classifier)

Clasificador corregido (dimensión 768): Linear(in_features=768, out_features=88, bias=True)


# **Fine-tuning**

In [41]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            features = model.encode_image(images)
            outputs = model.classifier(features)

            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

# Entrenamiento

In [47]:
import torch
import torch.nn as nn

# Nos aseguramos de que el clasificador esté definido con la dimensión de entrada correcta (512)
model.classifier = nn.Linear(512, len(classes)).to(device)
for param in model.classifier.parameters():
    param.requires_grad = True

# Volvemos a configurar el optimizador para apuntar a los parámetros del clasificador correcto
optimizer = torch.optim.AdamW(
    [
        {
            "params": model.visual.transformer.resblocks[-2:].parameters(),
            "lr": 1e-5
        },
        {
            "params": model.visual.ln_post.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.classifier.parameters(),
            "lr": 1e-3
        }
    ],
    weight_decay=1e-4
)

EPOCHS = 10
best_val_loss = float("inf")

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

for epoch in range(EPOCHS):

    train_loss, train_acc = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    val_loss, val_acc = run_epoch(
        model,
        val_loader,
        criterion
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Época {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/seed_bioclip_finetuned.pth"
        )

        print("  ✓ Mejor modelo guardado")

KeyboardInterrupt: 